In [19]:
# Imports
import os
import pandas as pd
pd.set_option('display.max_columns', None)
from pyspark.sql import SparkSession
from pathlib import Path
import numpy as np
from optbinning import OptimalBinning

# Setup 
project_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "data" / "processed" / "transactions_parquet").is_dir()
    ),
    None,
)

spark = SparkSession.builder \
        .appName("Real-Time Fraud Transaction Detection - Feature Selection") \
        .getOrCreate()

# Configuration
processed_data_path = project_root / "data" / "processed" / "transactions_parquet"
spark.sparkContext.setLogLevel("WARN")
TARGET = "is_fraud"

In [11]:
source_df = spark.read.parquet(str(processed_data_path)).toPandas()
print(f"end=> source_df.shape: {source_df.shape}")

end=> source_df.shape: (326799, 28)


## 1. Fill Rate Analysis

In [12]:
FR_THRESHOLD = 0.9

fr_report = pd.DataFrame({
    'COLUMN_NAME': source_df.columns,
    'TOTAL_COUNT': source_df.count(),
    'NON_MISSING_COUNT': source_df.notnull().sum().values,
    'FILL_RATE': source_df.notnull().sum().values / source_df.count(),
}).sort_values(by='FILL_RATE', ascending=False)

fr_report['DROP_FLAG'] = np.where(fr_report['FILL_RATE'] > FR_THRESHOLD, "KEEP", "DROP")

print(f"Dropped {len(fr_report[fr_report['DROP_FLAG'] == 'DROP'])} columns with fill rate below {FR_THRESHOLD}.")
fr_report

Dropped 0 columns with fill rate below 0.9.


,COLUMN_NAME,TOTAL_COUNT,NON_MISSING_COUNT,FILL_RATE,DROP_FLAG
trans_date_trans_time,trans_date_trans_time,326799,326799,1.0,KEEP
cc_num,cc_num,326799,326799,1.0,KEEP
merchant,merchant,326799,326799,1.0,KEEP
category,category,326799,326799,1.0,KEEP
amt,amt,326799,326799,1.0,KEEP
first,first,326799,326799,1.0,KEEP
last,last,326799,326799,1.0,KEEP
gender,gender,326799,326799,1.0,KEEP
street,street,326799,326799,1.0,KEEP
city,city,326799,326799,1.0,KEEP


## 2. Near Zero Variance

In [13]:
NZV_THRESHOLD = 0.05

nzv_report = pd.DataFrame({
    'COLUMN_NAME': source_df.columns,
    'MOST_FREQ_VALUE': [source_df[col].value_counts().sort_values(ascending=False).keys()[0] for col in source_df.columns],
    'MOST_FREQ_VALUE_COUNT': [source_df[col].value_counts().sort_values(ascending=False).values[0] for col in source_df.columns],
    'MOST_FREQ_VALUE_PORTION': [round(source_df[col].value_counts().sort_values(ascending=False).values[0] / len(source_df), 5) for col in source_df.columns]
})

nzv_report['DRP_FLAG'] = np.where(nzv_report['MOST_FREQ_VALUE_PORTION'] < 1 - NZV_THRESHOLD, "KEEP", "DROP")
print(f"Dropped {len(nzv_report[nzv_report['DRP_FLAG'] == 'DROP'])} columns with most frequent value portion above {NZV_THRESHOLD}.")
nzv_report

Dropped 1 columns with most frequent value portion above 0.05.


,COLUMN_NAME,MOST_FREQ_VALUE,MOST_FREQ_VALUE_COUNT,MOST_FREQ_VALUE_PORTION,DRP_FLAG
0,trans_date_trans_time,2019-05-31 11:11:54,8,0.00002,KEEP
1,cc_num,6011109736646996,845,0.00259,KEEP
2,merchant,fraud_Kilback LLC,1070,0.00327,KEEP
3,category,gas_transport,33599,0.10281,KEEP
4,amt,1.14,149,0.00046,KEEP
5,first,Christopher,6959,0.02129,KEEP
6,last,Smith,7113,0.02177,KEEP
7,gender,F,178506,0.54623,KEEP
8,street,594 Berry Lights Apt. 392,845,0.00259,KEEP
9,city,Birmingham,1383,0.00423,KEEP


## 3. IV filtering

In [14]:
IV_THRESHOLD = 0.2
iv_report = []

for col in source_df.columns:
    x = source_df[col]
    y = source_df[TARGET]

    opt = OptimalBinning(
        name=col, 
        dtype= 'numerical' if pd.api.types.is_numeric_dtype(x) else 'categorical'
    )

    opt.fit(x, y)
    iv = opt.binning_table.build()["IV"].sum()
    iv_report.append({
            "COLUMN_NAME": col,
            "IV": iv,
            "DROP_FLAG": "DROP" if iv < IV_THRESHOLD else "KEEP"
        })

iv_report = pd.DataFrame(iv_report).sort_values(
    "IV",
    ascending=False
)

print(f"Total Features      : {len(iv_report)}")
print(f"Features To Keep    : {(iv_report['DROP_FLAG'] == 'KEEP').sum()}")
print(f"Features To Drop    : {(iv_report['DROP_FLAG'] == 'DROP').sum()}")
iv_report

Total Features      : 28
Features To Keep    : 17
Features To Drop    : 11


,COLUMN_NAME,IV,DROP_FLAG
1,cc_num,9.538938,KEEP
8,street,9.538938,KEEP
11,zip,9.243730,KEEP
16,dob,8.695326,KEEP
26,amt_log1p,8.645804,KEEP
4,amt,8.645804,KEEP
15,job,8.424943,KEEP
9,city,7.476014,KEEP
6,last,6.976269,KEEP
5,first,6.025459,KEEP


## Combined Final Report

In [21]:
feature_selection_report = (
    fr_report[["COLUMN_NAME", "FILL_RATE", "DROP_FLAG"]]
    .rename(columns={"DROP_FLAG": "FILL_RATE_FLAG"})
    .merge(
        nzv_report[["COLUMN_NAME", "MOST_FREQ_VALUE_PORTION", "DRP_FLAG"]]
        .rename(columns={"DRP_FLAG": "NZV_FLAG"}),
        on="COLUMN_NAME",
        how="outer",
    )
    .merge(
        iv_report[["COLUMN_NAME", "IV", "DROP_FLAG"]]
        .rename(columns={"DROP_FLAG": "IV_FLAG"}),
        on="COLUMN_NAME",
        how="outer",
    )
)

target_mask = feature_selection_report["COLUMN_NAME"] == TARGET
feature_selection_report.loc[target_mask, "IV_FLAG"] = "NOT_APPLICABLE_TARGET"
feature_selection_report["FINAL_FLAG"] = np.where(
    target_mask
    | (
        (feature_selection_report["FILL_RATE_FLAG"] != "DROP")
        & (feature_selection_report["NZV_FLAG"] != "DROP")
        & (feature_selection_report["IV_FLAG"] != "DROP")
    ),
    "KEEP",
    "DROP",
)
feature_selection_report["ROLE"] = np.where(target_mask, "TARGET", "FEATURE")
feature_selection_report = feature_selection_report.sort_values(
    ["FINAL_FLAG", "ROLE", "COLUMN_NAME"]
).reset_index(drop=True)

feature_selection_report_path = "feature_selection_report"

feature_selection_report.to_csv(
    os.path.join(feature_selection_report_path, "feature_selection_report.csv"), index=False
)

print(f"Feature selection report: {os.path.join(feature_selection_report_path, "feature_selection_report.csv")}")
print(f"Features kept: {(feature_selection_report['FINAL_FLAG'] == 'KEEP').sum()}")
print(f"Features dropped: {(feature_selection_report['FINAL_FLAG'] == 'DROP').sum()}")
feature_selection_report

Feature selection report: feature_selection_report\feature_selection_report.csv
Features kept: 18
Features dropped: 10


,COLUMN_NAME,FILL_RATE,FILL_RATE_FLAG,MOST_FREQ_VALUE_PORTION,NZV_FLAG,IV,IV_FLAG,FINAL_FLAG,ROLE
0,distance_km,1.0,KEEP,0.00006,KEEP,0.014568,DROP,DROP,FEATURE
1,gender,1.0,KEEP,0.54623,KEEP,0.003705,DROP,DROP,FEATURE
2,is_weekend,1.0,KEEP,0.67696,KEEP,0.079010,DROP,DROP,FEATURE
3,lat,1.0,KEEP,0.00281,KEEP,0.137712,DROP,DROP,FEATURE
4,long,1.0,KEEP,0.00281,KEEP,0.132257,DROP,DROP,FEATURE
5,merch_lat,1.0,KEEP,0.00002,KEEP,0.067520,DROP,DROP,FEATURE
6,merch_long,1.0,KEEP,0.00002,KEEP,0.067080,DROP,DROP,FEATURE
7,trans_date_trans_time,1.0,KEEP,0.00002,KEEP,0.000000,DROP,DROP,FEATURE
8,trans_num,1.0,KEEP,0.00002,KEEP,0.000000,DROP,DROP,FEATURE
9,transaction_day_of_week,1.0,KEEP,0.18547,KEEP,0.134815,DROP,DROP,FEATURE
